<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/blind_set_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 54.3 MB/s eta 0:00:00


In [9]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Load the blind set CSV
csv_path = "blindset.xlsx"
df = pd.read_excel(csv_path)

# 2. Setup the SDWriter for the unbundled conformers
writer = Chem.SDWriter("BlindSet_3D.sdf")

for index, row in df.iterrows():
    smiles = row['Smiles']
    mol_id = str(row['ID'])

    # Generate 2D graph
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Failed to parse SMILES for ID {mol_id}")
        continue

    # Add explicit hydrogens
    mol = Chem.AddHs(mol)
    mol.SetProp("_Name", f"Compound_{mol_id}")

    # ETKDGv3 Embedding Setup
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    # Generate 5 conformers per unique molecule
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params)

    # Force Field Optimization and Spatial Isolation
    for cid in conf_ids:
        # MMFF94 Optimization with 500 max iterations
        try:
            AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=500)
        except Exception as e:
            print(f"Optimization failed for ID {mol_id}, Conf {cid}: {e}")
            continue

        # Unbundling algorithm: isolate each conformer into a clean object
        single_conf_mol = Chem.Mol(mol)
        single_conf_mol.RemoveAllConformers()
        single_conf_mol.AddConformer(mol.GetConformer(cid), assignId=True)

        # Save to SDF
        writer.write(single_conf_mol)

writer.close()
print("3D Embedding and Optimization Complete. Saved to BlindSet_3D.sdf")

3D Embedding and Optimization Complete. Saved to BlindSet_3D.sdf
